In [11]:
# ETL Timesheet
# Extração
## API
import requests
import json
# Tratamento
import pandas as pd
import numpy as np
## Sistemas e Data
from datetime import datetime, timedelta
import os
from urllib.parse import quote_plus
import urllib3
# Carga
import pyarrow

pc_user = os.getlogin()

In [12]:
# VARIÁVEIS

# Dados API-----------------------------------------------------------------------------------------------------------------------------------------
token = '4ihT[soma]tYzcMDUTILK2cQXHJFVPJqdiXTT0RZSKWnnymo9AeF04hQVm6w8yAc1Fno[soma]a/mbsX32zfJLCqHY9MntIg=='

url = 'https://espaider.com.br/Arauz/WCF/WCFExportaDados/WCFExportaDados.svc/ExportaDados'

identificador = 'BI_SOLICITACOES_SUPORTEESPAIDER'

# Suprime os avisos de requisições HTTPS não verificadas
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [13]:
def api_espaider_exportacao(
    url, params=None, 
    headers=None,
    body=None,
    data_inicio=None, 
    data_fim=None, 
    limite_registros=None):
    
    try:
        dfs_p = []
        pagina = 'inicio'
        contador = 0
        total_registros = 0
        print('Lendo registros.')

        while pagina:
            print(f'Página: {contador}')
            
            if pagina == 'inicio':
                
                if body == None:
                    response = requests.post(url, params=params,headers=headers, verify=False)
                
                else:
                    response = requests.post(url, params=params,headers=headers, json=body, verify=False)
            
            else:
                response = requests.get(pagina, verify=False)

            json_data = response.json()

            if json_data.get('Situacao') == "S":
                registros = json_data.get('ListaRegistros', [])

                dados = []
                for registro in registros:
                    linha = {'IDEspaider': registro.get('IDEspaider')}
                    linha.update({campo['Identificador']: campo['Valor'] for campo in registro['ListaCampos']})
                    dados.append(linha)

                if not dados:
                    break

                colunas = list(dados[0].keys())
                df_p = pd.DataFrame(dados, columns=colunas)
                dfs_p.append(df_p)
                total_registros += len(df_p)
                contador += 1

                if limite_registros and total_registros >= limite_registros:
                    break

                pagina = json_data.get('URLPaginacao')
            else:
                print(f"⚠ Erro no retorno da API: {json_data.get('MensagemRetorno')}")
                break

        df_pai = pd.concat(dfs_p, ignore_index=True)

        if limite_registros:
            df_pai = df_pai.iloc[:limite_registros]

        return df_pai

    except Exception as ex:
        print('⚠ Erro:', ex)
        return None


In [ ]:
# CONSULTA

# Headers da requisição
headers = {
    'Content-Type': 'application/json'
    }

params = {
    'token': token,
    'identificador': identificador
}

df = api_espaider_exportacao(url=url,params=params,headers=headers, limite_registros=10)

Lendo registros.
Página: 0


In [15]:
df.head()

,IDEspaider,VENCIMENTOPARCELA,VALOR,SITUACAOATUAL,REMETENTE,PLANO,PASTAPROVIDENCIA,PASTACONTENCIOSO,PARCELAREFERENCIA,NUMERODESTINATARIO,...,CORREIOTEXTOAR,CONTRATO,CODIGO,CNPJREMETENTE,CNPJAPRESENTANTE,CEPREMETENTE,CEPDESTINATARIO,BAIRRODESTINATARIO,APRESENTANTE,ADVERSO
0,2870976,10/10/2025,"1.563,00",Em aprovação,BANCO SAFRA S.A,36,,PRO.27893,18,,...,,B805218066,NOTIF.00013/25,58.160.789/0001-28,05.417.802/0001-15,-,,,ARAÚZ E ADVOGADOS,Ezequel Alves De Oliveira
1,2870977,06/10/2025,"2.653,00",Em aprovação,BANCO SAFRA S.A,24,,PRO.01585,12,123,...,,2015300616,NOTIF.00014/25,58.160.789/0001-28,05.417.802/0001-15,,86050-460,Jardim Petrópolis,ARAÚZ E ADVOGADOS,Cleiton Luiz De Paiva


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   IDEspaider              2 non-null      int64  
 1   VENCIMENTOPARCELA       2 non-null      object 
 2   VALOR                   2 non-null      object 
 3   SITUACAOATUAL           2 non-null      object 
 4   REMETENTE               2 non-null      object 
 5   PLANO                   2 non-null      object 
 6   PASTAPROVIDENCIA        0 non-null      float64
 7   PASTACONTENCIOSO        2 non-null      object 
 8   PARCELAREFERENCIA       2 non-null      object 
 9   NUMERODESTINATARIO      1 non-null      object 
 10  NOMEDESTINATARIO        1 non-null      object 
 11  MUNICIPIODESTINATARIO   1 non-null      object 
 12  LOGRADOURODESTINATARIO  1 non-null      object 
 13  IDENTIFICADOR           2 non-null      object 
 14  ESTADODESTINATARIO      1 non-null      object

In [ ]:
# df.to_excel('SOLICITACOES_NOTIFICACOES.xlsx', index=False)